 #  This MHW pipeline detect and calculte the MHW using 90th percentile threshold which is standered MHW detection {Hobdey et at. (2016)} method. If User wants to calculte the MHW using 80th percentile threshold, then Just change itr to "80" in the MHW detection code.

# Run the following script to check if all dependencies are installed and working

In [ ]:
# primary requirements are the libraries and packages must be installed (python3, numpy, pandas, xarray, dask, bottleneck, netCDF4, cftime, matplotlib, marineHeatWaves)
# also fixes np.NaN to np.nan in marineHeatWaves module

import sys, platform
import numpy as np
import pandas as pd
import xarray as xr, dask, bottleneck, netCDF4, cftime
from pathlib import Path
from typing import List, Optional, Tuple, Dict, Union
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
import marineHeatWaves as mhw, re, pathlib
from importlib.metadata import version, PackageNotFoundError

# convert np.NAN to np.nan to avoid dependency issues
if not hasattr(np, "NaN"):
    np.NaN = np.nan  

# Check the package is working or not by printing the versions of the dependencies
try:
    mhw_ver = version('marineHeatWaves')
except PackageNotFoundError:
    mhw_ver = getattr(mhw, '__version__', 'unknown')
print('OK:', pd.__version__, xr.__version__, dask.__version__, 'mhw:', mhw_ver)
print('mhw module file:', getattr(mhw, '__file__', 'unknown'))

# check the marineHeatWaves module file and replace np.NaN with np.nan
p = pathlib.Path(mhw.__file__)
p.write_text(re.sub(r'\bnp\.NaN\b', 'np.nan', p.read_text()))

print(sys.executable)
print(platform.python_version())


In [ ]:
# check if marineHeatWaves package is working or not by running a simple detection test

t = pd.date_range("2000-01-01", periods=30, freq="D") # a sample date range of 30 days
ords = np.array([d.toordinal() for d in t], dtype=int)
temp = 25 + np.sin(np.linspace(0, 6.28, 40)); temp[15:25] += 2

res, clim = mhw.detect(ords, temp, climatologyPeriod=[2000, 2000], pctile=90, minDuration=5, joinAcrossGaps=True) # Hobdey's MHW defination
print("events:", len(res.get("time_start", [])))

# Verifies that core libs are importable and compatible with marineHeatWaves.
from importlib import import_module
import numpy as np
import marineHeatWaves as mhw

libs = ["numpy", "pandas", "xarray", "dask", "bottleneck","netCDF4", "cftime", "marineHeatWaves"]
vers = {}
problems = []

for m in libs:
    try:
        import_module(m)
        try:
            vers[m] = version(m)
        except PackageNotFoundError:
            mod = import_module(m)
            vers[m] = getattr(mod, "__version__", "unknown")
    except Exception as e:
        problems.append((m, str(e)))
        vers[m] = "IMPORT FAILED"

print("Package versions detailes:")
for k in libs:
    print(f"{k:16s} : {vers[k]}")
if problems:
    print("\n Some packages failed to import:")
    for m, msg in problems:
        print(f" - {m}: {msg}")

# marineHeatWaves.detect accepts ordinal ints + 1D temp array
try:
    # make a synthetic 30-day series with a warm spell
    days = np.array([pd.Timestamp("2000-01-01") + pd.Timedelta(i, "D") for i in range(30)])
    ords = np.array([d.toordinal() for d in days], dtype=int)
    temp = 25 + np.sin(np.linspace(0, 6.28, 30))
    temp[15:25] += 2.0  # warm event
    res, clim = mhw.detect(ords, temp, time_period=[1991, 2020], pctile=90, minDuration=5, joinAcrossGaps=True)
    print("\n marineHeatWaves detection package is working correctly.")
    print(f"Detected events: {len(res.get('time_start', []))}")
except Exception as e:
    print("\n marineHeatWaves detection failed:", e)


In [ ]:
# Checking the files and Summarize the data

from glob import glob
import numpy as np, pandas as pd

# file path
sst_data = "/home/Desktop/Noah_data_1982-2024_SST_daily_mean/sst.day.mean.*.nc"

# region of interest
ROI = dict(lat_min=0.0, lat_max=30.0, lon_min=40.0, lon_max=110.0)

# specifies regions here
REGIONS = {
    "Arabian Sea":    {"lon_min": 40.0, "lon_max": 78.0,  "lat_min": 0.0, "lat_max": 30.0},
    "Bay Of Bengal":   {"lon_min": 78.0, "lon_max": 110.0, "lat_min": 0.0, "lat_max": 30.0},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0, "lat_min": 0.0, "lat_max": 30.0},
}

# data loading parameters
use_dask = True                 # set False to fully load into memory
time_chunk = {"time": 90} if use_dask else None
sample_map_date: Optional[str] = None  
var_name = "sst"       
time_period = [1982, 2024]  

# loading the data
def open_sst(files, time_chunk=None, engine: str = "netcdf4") -> xr.Dataset:
    if isinstance(files, str):
        p = Path(files)
        if any(ch in files for ch in "*?[]"):
            paths = sorted(glob(files))
            if not paths:
                raise FileNotFoundError(f"No files match glob: {files}")
            return xr.open_mfdataset(paths, combine="by_coords", parallel=True, time_chunk=time_chunk, engine=engine)
        else:
            if not p.exists():
                raise FileNotFoundError(f"File not found: {files}")
            return xr.open_dataset(files, time_chunk=time_chunk, engine=engine)
    else:
        paths = [str(Path(f)) for f in files]
        for f in paths:
            if not Path(f).exists():
                raise FileNotFoundError(f"File not found: {f}")
        return xr.open_mfdataset(paths, combine="by_coords", parallel=True, time_chunk=time_chunk, engine=engine)

# Subset to region of interest
def subset_roi(ds: xr.Dataset, roi: Dict[str, float], var: str = "sst") -> xr.Dataset:
    # Make sure coords are named commonly
    lat_name = "lat" if "lat" in ds.coords else "latitude"
    lon_name = "lon" if "lon" in ds.coords else "longitude"

    ds2 = ds.sel(**{lat_name: slice(roi["lat_min"], roi["lat_max"]),lon_name: slice(roi["lon_min"], roi["lon_max"]),})
    # Keep only the variable of interest and coords
    if var in ds2:
        return ds2[[var]]
    else:
        raise KeyError(f"Variable '{var}' not found. Available: {list(ds2.data_vars)}")

# Print metadata
def print_metadata(ds: xr.Dataset, var: str = "sst") -> None:
    # Dimention sizes
    for c in ds.coords:
        arr = ds.coords[c]
        try:
            vals = arr.values
            preview = f"{vals[:3]} ... {vals[-3:]}" if vals.size > 6 else vals
        except Exception:
            preview = arr
        print(f"{c}: {preview}")

    # convert time 
    if "time" in ds.coords:
        t = pd.to_datetime(ds["time"].values)
        print("\nTime Coverage:")
        print(f"Start: {pd.Timestamp(t[0]).date()}, End: {pd.Timestamp(t[-1]).date()}, Length: {t.size} steps")

    # Global attrs
    print("\nGlobal attributes:")
    for k, v in ds.attrs.items():
        print(f"{k}: {v}")

    # Variable attrs
    if var in ds:
        print(f"\nVariable '{var}' attributes:")
        for k, v in ds[var].attrs.items():
            print(f"{k}: {v}") 
            print("Dimensions:")
    for k, v in ds.dims.items():
        print(f"{k}: {v}")

    # Coordinates preview
    print("\nCoordinates (first few):")
   
def summarize_events_table(res: dict) -> pd.DataFrame:
    """
    Convert marineHeatWaves 'res' dict to a tidy event table.'time_*' in res are *ordinal* days (Python datetime.toordinal).
    """
    to_ts = lambda arr: pd.to_datetime([pd.Timestamp.fromordinal(int(d)) for d in np.asarray(arr)])

    return pd.DataFrame({
        "start_date":                   to_ts(res["time_start"]),
        "end_date":                     to_ts(res["time_end"]),
        "duration_days":                res["duration"],
        "intensity_max_degC":           res["intensity_max"],
        "intensity_mean_degC":          res["intensity_mean"],
        "cumulative_intensity_degC":    res["intensity_cumulative"],
    })

# Open files
ds = open_sst(sst_data, time_chunk={"time": 120}, engine="netcdf4")

# Subset ROI &  'sst'
ds_roi = subset_roi(ds, ROI, var=var_name)
da = ds_roi[var_name]

# Inspect metadata
print_metadata(ds_roi, var=var_name)


# Common script to calculate the marine heat wave for Arabian Sea, Bay of Bengal and North India Ocean

In [ ]:
# common script to calculate the marine heat wave 
# import the packages and libraries
from glob import glob
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar
import marineHeatWaves as mhw

# SST files path
FILES_GLOB = "/home/Desktop/Noah_data_1982-2024_SST_daily_mean/sst.day.mean.*.nc"
SST_VAR    = "sst"

# define the baseline year
BASELINE = (1982, 2024)

# Event definition based on Hobday et al. (2016)
MIN_DUR  = 5     # minimum event duration (days)
MAX_GAP  = 2     # join across gaps up to this many days

# Dask-friendly chunks
CHTIME, CHXY = 160, 80
 
# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}

OUTROOT = Path("outputs_mhw_without_avg"); OUTROOT.mkdir(parents=True, exist_ok=True)

# open SST data using xarray
def open_sst(files_glob: str, roi: dict) -> tuple[xr.Dataset, str, str]:
    paths = sorted(glob(files_glob))
    if not paths:
        raise FileNotFoundError(f"No files match: {files_glob}")
    ds = xr.open_mfdataset(paths, combine="by_coords", chunks={"time": CHTIME}, engine="netcdf4")
    latn = "lat" if "lat" in ds.coords else "latitude"
    lonn = "lon" if "lon" in ds.coords else "longitude"
    ds = ds.sel({latn: slice(roi["lat_min"], roi["lat_max"]), lonn: slice(roi["lon_min"], roi["lon_max"])})
    return ds, latn, lonn


# use the hobdey's defination on region
def _detect_mask_1d(sst_1d: np.ndarray, thresh_1d: np.ndarray, min_dur: int = MIN_DUR, max_gap: int = MAX_GAP) -> np.ndarray:
    
    ok = np.isfinite(sst_1d) & np.isfinite(thresh_1d)
    exc = ok & (sst_1d >= thresh_1d)
    if not exc.any():
        return exc

    x = exc.astype(np.int8)
    n = x.size

    # join short gaps
    i = 0
    while i < n:
        if x[i] == 1:
            j = i + 1
            while j < n and x[j] == 1:
                j += 1
            g0 = j
            while g0 < n and x[g0] == 0:
                g0 += 1
            gap_len = g0 - j
            if gap_len > 0 and gap_len <= max_gap:
                x[j:g0] = 1
                j = g0
            i = j
        else:
            i += 1

    # enforce min duration
    y = x.copy()
    i = 0
    while i < n:
        if y[i] == 1:
            j = i + 1
            while j < n and y[j] == 1:
                j += 1
            if (j - i) < min_dur:
                y[i:j] = 0
            i = j
        else:
            i += 1

    return y.astype(bool)

def detect_mask_time(sst: xr.DataArray, thresh: xr.DataArray) -> xr.DataArray:
    
    return xr.apply_ufunc(_detect_mask_1d, sst, thresh, input_core_dims=[["time"], ["time"]], output_core_dims=[["time"]], vectorize=True, dask="parallelized", output_dtypes=[bool])

# per grid climatology & threshold 
def _clim_thresh_time_1d(ords_1d: np.ndarray, temp_1d: np.ndarray, y0: int, y1: int, pct: int, min_dur: int, max_gap: int) -> tuple[np.ndarray, np.ndarray]:
    
    # If mostly missing, return NaNs to avoid unstable fits
    if np.isfinite(temp_1d).sum() < 30:
        n = temp_1d.size
        return np.full(n, np.nan, float), np.full(n, np.nan, float)

    # MHW detection
    _, clim = mhw.detect(ords_1d.astype(int), temp_1d.astype(float),climatologyPeriod=[int(y0), int(y1)], pctile=int(pct), minDuration=int(min_dur),
                         joinAcrossGaps=True, maxGap=int(max_gap),)
    seas   = np.asarray(clim["seas"],   float)
    thresh = np.asarray(clim["thresh"], float)
    return seas, thresh

# compute time-aligned climatology and threshold for every grid
def build_grid_baseline(ds: xr.Dataset, latn: str, lonn: str,pctile: int = 90) -> tuple[xr.DataArray, xr.DataArray]:
    
    t_index = pd.to_datetime(ds["time"].values)
    y0 = max(BASELINE[0], t_index.year.min())
    y1 = min(BASELINE[1], t_index.year.max())
    ords_da = xr.DataArray(np.array([d.toordinal() for d in t_index], dtype=int), coords={"time": ds["time"]}, dims=["time"],).chunk({"time": -1})  # single time chunk for gufunc

    sst = ds[SST_VAR].chunk({"time": -1, latn: CHXY, lonn: CHXY})

    seas_t, thresh_t = xr.apply_ufunc(_clim_thresh_time_1d, ords_da, sst,input_core_dims=[["time"], ["time"]], output_core_dims=[["time"], ["time"]], vectorize=True, dask="parallelized",
                                      output_dtypes=[float, float], kwargs=dict(y0=int(y0), y1=int(y1), pct=int(pctile), min_dur=MIN_DUR, max_gap=MAX_GAP))
    seas_t   = seas_t.rename("seas_t")
    thresh_t = thresh_t.rename("thresh_t")
    return seas_t, thresh_t


**Arabian Sea MHW detection per grid**

In [ ]:
# This script detect and save the per-grid MHW and save in zarr files for Arabian Sea
REG = "Arabian Sea"
ROI = ROI_DICT[REG]

#  Open SST data
ds, latn, lonn = open_sst(FILES_GLOB, ROI)

#  Land mask & chunks
ocean = ds[SST_VAR].notnull().any("time")
sst   = ds[SST_VAR].where(ocean).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

#  Per-grid climatology & 90th-threshold 
seas_t, thresh_t = build_grid_baseline(ds, latn, lonn, pctile=90)

# Detect Hobday events per grid 
# compute boolean mask for logic
evt_mask_bool = detect_mask_time(sst.chunk({"time": -1}),thresh_t.chunk({"time": -1})).rename("mhw_mask")  # bool

# NaN on land & keep float so NaNs persist to Zarr
evt_mask = evt_mask_bool.where(ocean).astype("float32")


# Daily metrics for future analysis
intensity = (sst - seas_t).rename("intensity")                
excess    = (sst - thresh_t).where(evt_mask == 1).rename("excess")  # only on event days

#  Yearly per-grid summaries
# compute starts on the boolean mask
starts_bool = (evt_mask_bool & ~(evt_mask_bool.shift(time=1, fill_value=False)))

# NaN on land, keep float dtype (so NaNs survive)
starts = starts_bool.where(ocean).astype("float32")

events_per_year_grid = (starts.groupby("time.year").sum("time").where(ocean).rename("events_per_year").astype("float32"))  
days_per_year_grid = (evt_mask.groupby("time.year").sum("time").where(ocean).rename("days_per_year").astype("float32"))
mean_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").mean("time").rename("mean_intensity_year").astype("float32"))
max_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").max("time").rename("max_intensity_year").astype("float32"))
cumulative_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").sum("time").rename("cumulative_intensity_year").astype("float32"))

# SAVE (Zarr + CSV) 
out_dir = OUTROOT / ROI_DICT[REG]["slug"]; out_dir.mkdir(parents=True, exist_ok=True)

daily_ds = xr.Dataset({"mhw_mask": evt_mask, "intensity": intensity, "excess": excess,"seas_t": seas_t, "thresh_t": thresh_t},coords={"time": ds["time"], latn: ds[latn], lonn: ds[lonn]},
                       ).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

# Keep only core coords to avoid Zarr alignment issues
keep = {"time", latn, lonn}
dropc = [c for c in daily_ds.coords if c not in keep]
if dropc:
    daily_ds = daily_ds.reset_coords(dropc, drop=True)

print("Writing Zarr outputs… (can take time)")
with ProgressBar():
    daily_ds.to_zarr(out_dir / "mhw_daily.zarr", mode="w", safe_chunks=False, align_chunks=True)
    events_per_year_grid.to_zarr(out_dir / "mhw_events_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    days_per_year_grid.to_zarr(out_dir / "mhw_days_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    mean_intensity_year_grid.to_zarr(out_dir / "mhw_mean_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    max_intensity_year_grid.to_zarr(out_dir / "mhw_max_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    cumulative_intensity_year_grid.to_zarr(out_dir / "mhw_cumulative_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)

print("Arabian Sea — done.")


**Arabian Sea MHW charaterstics bar graph with trends plotting** 

In [ ]:
# This script plots the MHW event and days bar graphs with trend lines per decades and per year using MHW zarr data of Arabian Sea
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm

# Define the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "Arabian Sea"
ROI = ROI_DICT[REG]

# Load the zarr files
ds_event_path = xr.open_zarr("/home//Jupyter files/arabian_sea/mhw_events_per_year_grid.zarr")
ds_days_path = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_days_per_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_event_path.coords else "latitude"
lonn = "lon" if "lon" in ds_event_path.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
events_per_year = ds_event_path["events_per_year"]
days_per_year  =  ds_days_path["days_per_year"]



# Recreate the ocean mask (True for ocean, False for land)
ocean = events_per_year.notnull().any("year")

# X-axis year tick 
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Area-weighted regional means (per year)
freq_region = (events_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
days_region = (days_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

total_events_region = float(freq_region.sum().values)
total_days_region   = float(days_region.sum().values)

print(f"{REG} totals — events: {total_events_region:.1f}, days: {total_days_region:.1f}")

# Plot the graphs
years_freq = (freq_region.coords.get("year", None).values
              if "year" in freq_region.coords
              else freq_region.get_index("year").values).astype(int)
years_days = (days_region.coords.get("year", None).values
              if "year" in days_region.coords
              else days_region.get_index("year").values).astype(int)


In [ ]:
# Events and days per decade bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    # 1. OLS linear regression trend 
    slop_ols, intercept_ols, r_val_ols, p_ols, std_err_ols= stats.linregress(x,y)
    line_ols= (slop_ols*x) + intercept_ols
    # p value test
    p_ols_str = "<0.01" if p_ols< 0.01 else f"{p_ols:.3f}"
    label_ols =  f"Linear Regression Trend: +{slop_ols*10:.2f}/decade ($p={p_ols_str}$)"
    # plot the trend line 
    ax.plot(x, line_ols, color= "red", linestyle= '--', linewidth= 2, label= label_ols)
    
    # 2. Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # 3. Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per decade.
    slope_coef = glm_results.params[1]
    decadal_pct_increase = (np.exp(slope_coef * 10) - 1) * 100
    label_glm = f"Poisson GLM: +{decadal_pct_increase:.1f}% /decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "green", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# plot the Events bar chart with all trend
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(years_freq, freq_region.values, color="#5b8def", width=0.8)
# add trend line
plot_trendlines(ax1,years_freq,freq_region)
ax1.set_title(f" MHW Events Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("MHW Events Per Year", fontweight= "bold")
ax1.grid(True, ls="--", alpha=0.4)
ax1.text(0.5, 0.92, f"Total Events: {total_events_region:.1f}", transform=ax1.transAxes, ha="center")
apply_year_ticks(ax1, years_freq)  
plt.tight_layout() 
plt.show()

# plot the days bar chart with all trend
fig, ax2 = plt.subplots(figsize=(12, 4))
ax2.bar(years_days, days_region.values, color="#f6a300", width=0.8)
# add trend line
plot_trendlines(ax2,years_days, days_region)
ax2.set_title(f"MHW Days Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold"); ax2.set_ylabel("MHW Days Per Year", fontweight= "bold")
ax2.grid(True, ls="--", alpha=0.4)
ax2.text(0.5, 0.92, f"Total MHW days: {total_days_region:.1f}", transform=ax2.transAxes, ha="center")
apply_year_ticks(ax2, years_days) 
plt.tight_layout()
plt.show()


**To calculate the year trendline, just remove the 10 multiplication from the *plot_trendlines* fuctions in required trendline calculation and chamge the unit from decaede to year. Rest of the code stays same.**   
**This is valid for all the trendline plotting in the given code scripts**

**Kolmogorov-smirnove two sample test**

In [ ]:
# This script performs the Two-Sample Kolmogorov-Smirnov (K-S) Test to compare the distributions of MHW events and days between 
# two time periods (1982–2002) and (2003–2024).
import scipy.stats as stats
import numpy as np

#  Define the split year 
split_year = 2002

years_freq = (freq_region.coords.get("year", None).values
              if "year" in freq_region.coords
              else freq_region.get_index("year").values).astype(int)
years_days = (days_region.coords.get("year", None).values
              if "year" in days_region.coords
              else days_region.get_index("year").values).astype(int)


# Create boolean masks to split the data into two time periods (1982–2002 and 2003–2024)
mask_era1_event = (years_freq >= 1982) & (years_freq <= split_year)
mask_era2_event = (years_freq > split_year) & (years_freq <= 2024)

mask_era1_days = (years_days >= 1982) & (years_days <= split_year)
mask_era2_days = (years_days > split_year) & (years_days <= 2024)

# Extract the continuous data 
freq_region_era1 = freq_region.values[mask_era1_event]
freq_region_era2 = freq_region.values[mask_era2_event]
days_region_era1 = days_region.values[mask_era1_days]
days_region_era2 = days_region.values[mask_era2_days]

# Run the Two-Sample K-S Test

# for MHW Events
ks_statistic, p_value_ks = stats.ks_2samp(freq_region_era1, freq_region_era2)

print("Two-Sample Kolmogorov-Smirnov Test Results for MHW Events:")
print(f"Era 1 (1982-{split_year}) vs Era 2 ({split_year+1}-2024)")
print(f"K-S Statistic (D): {ks_statistic:.4f}")
print(f"P-value: {p_value_ks:.4e}")

if p_value_ks < 0.05:
    print("Conclusion: The distribution of MHW Events has shifted significantly between the 1982-2002 and 2003-2024 periods.")
else:
    print("Conclusion: No statistically significant shift in the overall distribution.")

# for MHW Days
ks_statistic, p_value_ks = stats.ks_2samp(days_region_era1, days_region_era2)

print("\nTwo-Sample Kolmogorov-Smirnov Test Results for MHW Days:")
print(f"Era 1 (1982-{split_year}) vs Era 2 ({split_year+1}-2024)")
print(f"K-S Statistic (D): {ks_statistic:.4f}")
print(f"P-value: {p_value_ks:.4e}")

if p_value_ks < 0.05:
    print("Conclusion: The distribution of MHW Days has shifted significantly between 1982-2002 and 2003-2024 periods.")
else:
    print("Conclusion: No statistically significant shift in the overall distribution.")

In [ ]:
# The script plots the MHW Mean, Max and Cumulative intensity bar with trend lined per decades and per year using MHW zarr data of Arabian Sea
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Defire the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "Arabian Sea"
ROI = ROI_DICT[REG]

# Load the zarr files
ds_mean_int = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_mean_intensity_year_grid.zarr")
ds_max_int= xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_max_intensity_year_grid.zarr")
ds_cum_int= xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_cumulative_intensity_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_mean_int.coords else "latitude"
lonn = "lon" if "lon" in ds_mean_int.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
mean_intensity_year  =  ds_mean_int["mean_intensity_year"]
max_intensity_year  =  ds_max_int["max_intensity_year"]
cumulative_intensity_year= ds_cum_int["cumulative_intensity_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = mean_intensity_year.notnull().any("year")

# X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Mean, max and cumulative intensity per year
mean_int_region = mean_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
max_int_region = max_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
cum_int_region = cumulative_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()

In [ ]:
# Per decade MHW intensity bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    # 1. OLS linear regression trend 
    slop_ols, intercept_ols, r_val_ols, p_ols, std_err_ols= stats.linregress(x,y)
    line_ols= (slop_ols*x) + intercept_ols
    # p value test
    p_ols_str = "<0.01" if p_ols< 0.01 else f"{p_ols:.3f}"
    label_ols =  f"Linear Regression Trend: +{slop_ols*10:.2f}/decade ($p={p_ols_str}$)"
    # plot the trend line 
    ax.plot(x, line_ols, color= "red", linestyle= '--', linewidth= 2, label= label_ols)
    
    # 2. Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # 3. Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per decade.
    slope_coef = glm_results.params[1]
    decadal_pct_increase = (np.exp(slope_coef * 10) - 1) * 100
    label_glm = f"Poisson GLM: +{decadal_pct_increase:.1f}% /decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "blue", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()


# Plotting the bar graphs

years_mean_int = (mean_int_region.coords.get("year", None).values
              if "year" in mean_int_region.coords
              else mean_int_region.get_index("year").values).astype(int)

# Mean intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, mean_int_region, color="#0072B2", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, mean_int_region)
ax.set_title(f"Mean MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Maximum intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, max_int_region, color="#D55E00", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, max_int_region)
ax.set_title(f"Maximum MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Cumulative intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, cum_int_region, color= "#009E73", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, cum_int_region)
ax.set_title(f"Cumulative MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()


**Bay of Bengal MHW detection per grid**

In [ ]:
# This script detect and save the per-grid MHW and save in zarr files for Bay of Bengal
REG = "Bay Of Bengal"
ROI = ROI_DICT[REG]

#  Open SST data
ds, latn, lonn = open_sst(FILES_GLOB, ROI)

#  Land mask & chunks
ocean = ds[SST_VAR].notnull().any("time")
sst   = ds[SST_VAR].where(ocean).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

#  Per-grid climatology & 90th-threshold 
seas_t, thresh_t = build_grid_baseline(ds, latn, lonn, pctile=90)

# Detect Hobday events per grid 
# compute boolean mask for logic
evt_mask_bool = detect_mask_time(sst.chunk({"time": -1}),thresh_t.chunk({"time": -1})).rename("mhw_mask")  # bool

# NaN on land & keep float so NaNs persist to Zarr
evt_mask = evt_mask_bool.where(ocean).astype("float32")


# Daily metrics for future analysis
intensity = (sst - seas_t).rename("intensity")                
excess    = (sst - thresh_t).where(evt_mask == 1).rename("excess")  # only on event days

#  Yearly per-grid summaries
# compute starts on the boolean mask
starts_bool = (evt_mask_bool & ~(evt_mask_bool.shift(time=1, fill_value=False)))

# NaN on land, keep float dtype
starts = starts_bool.where(ocean).astype("float32")

events_per_year_grid = (starts.groupby("time.year").sum("time").where(ocean).rename("events_per_year").astype("float32"))  
days_per_year_grid = (evt_mask.groupby("time.year").sum("time").where(ocean).rename("days_per_year").astype("float32"))
mean_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").mean("time").rename("mean_intensity_year").astype("float32"))
max_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").max("time").rename("max_intensity_year").astype("float32"))
cumulative_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").sum("time").rename("cumulative_intensity_year").astype("float32"))

# SAVE (Zarr + CSV) 
out_dir = OUTROOT / ROI_DICT[REG]["slug"]; out_dir.mkdir(parents=True, exist_ok=True)

daily_ds = xr.Dataset({"mhw_mask": evt_mask, "intensity": intensity, "excess": excess,"seas_t": seas_t, "thresh_t": thresh_t},coords={"time": ds["time"], latn: ds[latn], lonn: ds[lonn]},
                       ).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

# Keep only core coords to avoid Zarr alignment issues
keep = {"time", latn, lonn}
dropc = [c for c in daily_ds.coords if c not in keep]
if dropc:
    daily_ds = daily_ds.reset_coords(dropc, drop=True)

print("Writing Zarr outputs… (can take time)")
with ProgressBar():
    daily_ds.to_zarr(out_dir / "mhw_daily.zarr", mode="w", safe_chunks=False, align_chunks=True)
    events_per_year_grid.to_zarr(out_dir / "mhw_events_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    days_per_year_grid.to_zarr(out_dir / "mhw_days_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    mean_intensity_year_grid.to_zarr(out_dir / "mhw_mean_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    max_intensity_year_grid.to_zarr(out_dir / "mhw_max_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    cumulative_intensity_year_grid.to_zarr(out_dir / "mhw_cumulative_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)


print("Bay Of Bengal — done.")


**Bay of Bengal MHW charaterstics bar graph with trends plotting** 

In [ ]:
# This script plots the MHW event and days bar graphs with trend lines per decades and per year using MHW zarr data in Bay of Bengal
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Defire the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "Bay Of Bengal"
ROI = ROI_DICT[REG]

# Load the zarr files
ds_event_path = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_events_per_year_grid.zarr")
ds_days_path = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_days_per_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_event_path.coords else "latitude"
lonn = "lon" if "lon" in ds_event_path.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
events_per_year = ds_event_path["events_per_year"]
days_per_year  =  ds_days_path["days_per_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = events_per_year.notnull().any("year")

# # X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Area-weighted regional means (per year)
freq_region = (events_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
days_region = (days_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

total_events_region = float(freq_region.sum().values)
total_days_region   = float(days_region.sum().values)

print(f"{REG} totals — events: {total_events_region:.1f}, days: {total_days_region:.1f}")

# Plot the graphs
years_freq = (freq_region.coords.get("year", None).values
              if "year" in freq_region.coords
              else freq_region.get_index("year").values).astype(int)
years_days = (days_region.coords.get("year", None).values
              if "year" in days_region.coords
              else days_region.get_index("year").values).astype(int)

In [ ]:
# Per decade MHW event and days bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    # 1. OLS linear regression trend 
    slop_ols, intercept_ols, r_val_ols, p_ols, std_err_ols= stats.linregress(x,y)
    line_ols= (slop_ols*x) + intercept_ols
    # p value test
    p_ols_str = "<0.01" if p_ols< 0.01 else f"{p_ols:.3f}"
    label_ols =  f"Linear Regression Trend: +{slop_ols*10:.2f}/decade ($p={p_ols_str}$)"
    # plot the trend line 
    ax.plot(x, line_ols, color= "red", linestyle= '--', linewidth= 2, label= label_ols)
    
    # 2. Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # 3. Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per decade.
    slope_coef = glm_results.params[1]
    decadal_pct_increase = (np.exp(slope_coef * 10) - 1) * 100
    label_glm = f"Poisson GLM: +{decadal_pct_increase:.1f}% /decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "green", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# plot the Events bar chart with all trend
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(years_freq, freq_region.values, color="#5b8def", width=0.8)
# add trend line
plot_trendlines(ax1,years_freq,freq_region)
ax1.set_title(f" MHW Events Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("MHW Events Per Year", fontweight= "bold")
ax1.grid(True, ls="--", alpha=0.4)
ax1.text(0.5, 0.92, f"Total Events: {total_events_region:.1f}", transform=ax1.transAxes, ha="center")
apply_year_ticks(ax1, years_freq)  
plt.tight_layout() 
plt.show()

# plot the days bar chart with all trend
fig, ax2 = plt.subplots(figsize=(12, 4))
ax2.bar(years_days, days_region.values, color="#f6a300", width=0.8)
# add trend line
plot_trendlines(ax2,years_days, days_region)
ax2.set_title(f"MHW Days Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold"); ax2.set_ylabel("MHW Days Per Year", fontweight= "bold")
ax2.grid(True, ls="--", alpha=0.4)
ax2.text(0.5, 0.92, f"Total MHW days: {total_days_region:.1f}", transform=ax2.transAxes, ha="center")
apply_year_ticks(ax2, years_days) 
plt.tight_layout()
plt.show()

In [ ]:
# This script plots the MHW Mean, Max and Cumulative intensity bar graphs using MHW zarr data of Bay of Bengal
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Defire the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "Bay Of Bengal"
ROI = ROI_DICT[REG]

# Load the zarr files
ds_mean_int = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_mean_intensity_year_grid.zarr")
ds_max_int= xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_max_intensity_year_grid.zarr")
ds_cum_int= xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_cumulative_intensity_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_mean_int.coords else "latitude"
lonn = "lon" if "lon" in ds_mean_int.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
mean_intensity_year  =  ds_mean_int["mean_intensity_year"]
max_intensity_year  =  ds_max_int["max_intensity_year"]
cumulative_intensity_year= ds_cum_int["cumulative_intensity_year"]



# Recreate the ocean mask (True for ocean, False for land)
ocean = mean_intensity_year.notnull().any("year")

# X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Mean, max and cumulative intensity per year
mean_int_region = mean_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
max_int_region = max_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
cum_int_region = cumulative_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()


In [ ]:
# Per decade MHW Intensity bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    # 1. OLS linear regression trend 
    slop_ols, intercept_ols, r_val_ols, p_ols, std_err_ols= stats.linregress(x,y)
    line_ols= (slop_ols*x) + intercept_ols
    # p value test
    p_ols_str = "<0.01" if p_ols< 0.01 else f"{p_ols:.3f}"
    label_ols =  f"Linear Regression Trend: +{slop_ols*10:.2f}/decade ($p={p_ols_str}$)"
    # plot the trend line 
    ax.plot(x, line_ols, color= "red", linestyle= '--', linewidth= 2, label= label_ols)
    
    # 2. Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # 3. Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per decade.
    slope_coef = glm_results.params[1]
    decadal_pct_increase = (np.exp(slope_coef * 10) - 1) * 100
    label_glm = f"Poisson GLM: +{decadal_pct_increase:.1f}% /decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "blue", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# Plotting the bar graphs
years_mean_int = (mean_int_region.coords.get("year", None).values
              if "year" in mean_int_region.coords
              else mean_int_region.get_index("year").values).astype(int)

# Mean intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, mean_int_region, color="#0072B2", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, mean_int_region)
ax.set_title(f"Mean MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Maximum intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, max_int_region, color="#D55E00", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, max_int_region)
ax.set_title(f"Maximum MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Cumulative intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, cum_int_region, color= "#009E73", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, cum_int_region)
ax.set_title(f"Cumulative MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

**MHW detection per grid in North Indian Ocean**

In [ ]:
# T# This script detect and save the per-grid MHW and save in zarr files for North Indian Ocean
REG = "North Indian Ocean"
ROI = ROI_DICT[REG]

#  Open SST data
ds, latn, lonn = open_sst(FILES_GLOB, ROI)

#  Land mask & chunks
ocean = ds[SST_VAR].notnull().any("time")
sst   = ds[SST_VAR].where(ocean).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

#  Per-grid climatology & 90th-threshold 
seas_t, thresh_t = build_grid_baseline(ds, latn, lonn, pctile=90)

# Detect Hobday events per grid 
# compute boolean mask for logic
evt_mask_bool = detect_mask_time(sst.chunk({"time": -1}),thresh_t.chunk({"time": -1})).rename("mhw_mask")  # bool

# NaN on land & keep float so NaNs persist to Zarr
evt_mask = evt_mask_bool.where(ocean).astype("float32")


# Daily metrics for future analysis
intensity = (sst - seas_t).rename("intensity")                 
excess    = (sst - thresh_t).where(evt_mask == 1).rename("excess")  # only on event days

#  Yearly per-grid summaries
# compute starts on the boolean mask
starts_bool = (evt_mask_bool & ~(evt_mask_bool.shift(time=1, fill_value=False)))

# NaN on land, keep float dtype 
starts = starts_bool.where(ocean).astype("float32")

events_per_year_grid = (starts.groupby("time.year").sum("time").where(ocean).rename("events_per_year").astype("float32"))  
days_per_year_grid = (evt_mask.groupby("time.year").sum("time").where(ocean).rename("days_per_year").astype("float32"))
mean_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").mean("time").rename("mean_intensity_year").astype("float32"))
max_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").max("time").rename("max_intensity_year").astype("float32"))
cumulative_intensity_year_grid = (intensity.where(evt_mask == 1).groupby("time.year").sum("time").rename("cumulative_intensity_year").astype("float32"))

# SAVE (Zarr + CSV) 
out_dir = OUTROOT / ROI_DICT[REG]["slug"]; out_dir.mkdir(parents=True, exist_ok=True)

daily_ds = xr.Dataset({"mhw_mask": evt_mask, "intensity": intensity, "excess": excess,"seas_t": seas_t, "thresh_t": thresh_t},coords={"time": ds["time"], latn: ds[latn], lonn: ds[lonn]},
                       ).chunk({"time": CHTIME, latn: CHXY, lonn: CHXY})

# Keep only core coords to avoid Zarr alignment issues
keep = {"time", latn, lonn}
dropc = [c for c in daily_ds.coords if c not in keep]
if dropc:
    daily_ds = daily_ds.reset_coords(dropc, drop=True)

print("Writing Zarr outputs… (can take time)")
with ProgressBar():
    daily_ds.to_zarr(out_dir / "mhw_daily.zarr", mode="w", safe_chunks=False, align_chunks=True)
    events_per_year_grid.to_zarr(out_dir / "mhw_events_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    days_per_year_grid.to_zarr(out_dir / "mhw_days_per_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    mean_intensity_year_grid.to_zarr(out_dir / "mhw_mean_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    max_intensity_year_grid.to_zarr(out_dir / "mhw_max_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)
    cumulative_intensity_year_grid.to_zarr(out_dir / "mhw_cumulative_intensity_year_grid.zarr", mode="w", safe_chunks=False, align_chunks=True)


print("North Indian Ocean — done.")

**North Indian Ocean MHW charaterstics bar graph with trends plotting** 

In [ ]:
# This script plots the MHW event and days bar graphs using MHW zarr data with per decades and per year trends in North Indian Ocean
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Defire the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "North Indian Ocean"
ROI = ROI_DICT[REG]

# Load the zarr files
ds_event_path = xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_events_per_year_grid.zarr")
ds_days_path = xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_days_per_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_event_path.coords else "latitude"
lonn = "lon" if "lon" in ds_event_path.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
events_per_year = ds_event_path["events_per_year"]
days_per_year  =  ds_days_path["days_per_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = events_per_year.notnull().any("year")

# # X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Area-weighted regional means (per year)
freq_region = (events_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
days_region = (days_per_year.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

total_events_region = float(freq_region.sum().values)
total_days_region   = float(days_region.sum().values)

print(f"{REG} totals — events: {total_events_region:.1f}, days: {total_days_region:.1f}")

# Plot the graphs
years_freq = (freq_region.coords.get("year", None).values
              if "year" in freq_region.coords
              else freq_region.get_index("year").values).astype(int)
years_days = (days_region.coords.get("year", None).values
              if "year" in days_region.coords
              else days_region.get_index("year").values).astype(int)

In [ ]:
# Per decade MHW event and days bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    # 1. OLS linear regression trend 
    slop_ols, intercept_ols, r_val_ols, p_ols, std_err_ols= stats.linregress(x,y)
    line_ols= (slop_ols*x) + intercept_ols
    # p value test
    p_ols_str = "<0.01" if p_ols< 0.01 else f"{p_ols:.3f}"
    label_ols =  f"Linear Regression Trend: +{slop_ols*10:.2f}/decade ($p={p_ols_str}$)"
    # plot the trend line 
    ax.plot(x, line_ols, color= "red", linestyle= '--', linewidth= 2, label= label_ols)
    
    # 2. Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # 3. Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per decade.
    slope_coef = glm_results.params[1]
    decadal_pct_increase = (np.exp(slope_coef * 10) - 1) * 100
    label_glm = f"Poisson GLM: +{decadal_pct_increase:.1f}% /decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "green", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# plot the Events bar chart with all trend
fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(years_freq, freq_region.values, color="#5b8def", width=0.8)
# add trend line
plot_trendlines(ax1,years_freq,freq_region)
ax1.set_title(f" MHW Events Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("MHW Events Per Year", fontweight= "bold")
ax1.grid(True, ls="--", alpha=0.4)
ax1.text(0.5, 0.92, f"Total Events: {total_events_region:.1f}", transform=ax1.transAxes, ha="center")
apply_year_ticks(ax1, years_freq)  
plt.tight_layout() 
plt.show()

# plot the days bar chart with all trend
fig, ax2 = plt.subplots(figsize=(12, 4))
ax2.bar(years_days, days_region.values, color="#f6a300", width=0.8)
# add trend line
plot_trendlines(ax2,years_days, days_region)
ax2.set_title(f"MHW Days Per Year in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold"); ax2.set_ylabel("MHW Days Per Year", fontweight= "bold")
ax2.grid(True, ls="--", alpha=0.4)
ax2.text(0.5, 0.92, f"Total MHW days: {total_days_region:.1f}", transform=ax2.transAxes, ha="center")
apply_year_ticks(ax2, years_days) 
plt.tight_layout()
plt.show()

In [ ]:
# This script plots the MHW Mean, Max and Cumulative intensity bar graphs with per decades and per year trends in North Indian Ocean
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Defire the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "North Indian Ocean"
ROI = ROI_DICT[REG]

# Load the zarr files
ds_mean_int = xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_mean_intensity_year_grid.zarr")
ds_max_int= xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_max_intensity_year_grid.zarr")
ds_cum_int= xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_cumulative_intensity_year_grid.zarr")

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_mean_int.coords else "latitude"
lonn = "lon" if "lon" in ds_mean_int.coords else "longitude"

# Extract the specific variable array from the loaded Dataset
mean_intensity_year  =  ds_mean_int["mean_intensity_year"]
max_intensity_year  =  ds_max_int["max_intensity_year"]
cumulative_intensity_year= ds_cum_int["cumulative_intensity_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = mean_intensity_year.notnull().any("year")

# X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Mean, max and cumulative intensity per year
mean_int_region = mean_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
max_int_region = max_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
cum_int_region = cumulative_intensity_year.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()

In [ ]:
# Per decade MHW Intensity bar graphs with trends
def plot_trendlines(ax, x,y):
    y= np.asarray(y)
    # 1. OLS linear regression trend 
    slop_ols, intercept_ols, r_val_ols, p_ols, std_err_ols= stats.linregress(x,y)
    line_ols= (slop_ols*x) + intercept_ols
    # p value test
    p_ols_str = "<0.01" if p_ols< 0.01 else f"{p_ols:.3f}"
    label_ols =  f"Linear Regression Trend: +{slop_ols*10:.2f}/decade ($p={p_ols_str}$)"
    # plot the trend line 
    ax.plot(x, line_ols, color= "red", linestyle= '--', linewidth= 2, label= label_ols)
    
    # 2. Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$)"
    # plot the trend line 
    ax.plot(x, line_ts, color= "black", linestyle= '--', linewidth= 2, label= label_ts)
    
    # 3. Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per decade.
    slope_coef = glm_results.params[1]
    decadal_pct_increase = (np.exp(slope_coef * 10) - 1) * 100
    label_glm = f"Poisson GLM: +{decadal_pct_increase:.1f}% /decade ($p={p_glm_str}$)"
    ax.plot(x, line_glm, color= "blue", linestyle= '--', linewidth= 2, label= label_glm)

    # add legends
    ax.legend()

# Plotting the bar graphs
years_mean_int = (mean_int_region.coords.get("year", None).values
              if "year" in mean_int_region.coords
              else mean_int_region.get_index("year").values).astype(int)

# Mean intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, mean_int_region, color="#0072B2", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, mean_int_region)
ax.set_title(f"Mean MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Maximum intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, max_int_region, color="#D55E00", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, max_int_region)
ax.set_title(f"Maximum MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Cumulative intensity
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(years_mean_int, cum_int_region, color= "#009E73", width=0.8)
# add trend line
plot_trendlines(ax, years_mean_int, cum_int_region)
ax.set_title(f"Cumulative MHW Intensity in {REG} ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax.set_xlabel("Year", fontweight= "bold")
ax.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax, years_mean_int)
ax.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

**Following scripts plot the combined MHW charactertics with trends in all regions** 

In [ ]:
# This script plots the combined plot of MHW Mean, Max and Cumulative intensity bar graphs using MHW zarr data of Arabian Sea, bay of bengal and North Indian Ocean
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Defire the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "Arabian Sea"
ROI = ROI_DICT[REG]

# Load the zarr files of Arabian Sea
ds_mean_int_as = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_mean_intensity_year_grid.zarr")
ds_max_int_as= xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_max_intensity_year_grid.zarr")
ds_cum_int_as= xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_cumulative_intensity_year_grid.zarr")

# Load the zarr files of Bay of Bengal
ds_mean_int_bob = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_mean_intensity_year_grid.zarr")
ds_max_int_bob= xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_max_intensity_year_grid.zarr")
ds_cum_int_bob= xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_cumulative_intensity_year_grid.zarr")

# Load the Zarr files of North Indian Ocean
ds_mean_int_nio = xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_mean_intensity_year_grid.zarr")
ds_max_int_nio= xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_max_intensity_year_grid.zarr")
ds_cum_int_nio= xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_cumulative_intensity_year_grid.zarr")



# Corrected coordinate extraction
latn = "lat" if "lat" in ds_mean_int_as.coords else "latitude"
lonn = "lon" if "lon" in ds_mean_int_as.coords else "longitude"


# Extract the mean intensity variable array from the loaded dataset of Arabian Sea
mean_intensity_year_as  =  ds_mean_int_as["mean_intensity_year"]
max_intensity_year_as  =  ds_max_int_as["max_intensity_year"]
cumulative_intensity_year_as= ds_cum_int_as["cumulative_intensity_year"]

# Extract the max intensity variable array from the loaded dataset of Bay of Bengal
mean_intensity_year_bob  =  ds_mean_int_bob["mean_intensity_year"]
max_intensity_year_bob  =  ds_max_int_bob["max_intensity_year"]
cumulative_intensity_year_bob= ds_cum_int_bob["cumulative_intensity_year"]

# Extract the cumulative intensity variable array from the loaded dataset of North Indian Ocean
mean_intensity_year_nio  =  ds_mean_int_nio["mean_intensity_year"]
max_intensity_year_nio  =  ds_max_int_nio["max_intensity_year"]
cumulative_intensity_year_nio = ds_cum_int_nio["cumulative_intensity_year"]

# Recreate the ocean mask (True for ocean, False for land)
ocean = mean_intensity_year.notnull().any("year")

# X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Mean, max and cumulative intensity per year in Arabian Sea
mean_int_region_as = mean_intensity_year_as.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
max_int_region_as = max_intensity_year_as.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
cum_int_region_as = cumulative_intensity_year_as.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()

# Mean, max and cumulative intensity per year in Bay of Bengal
mean_int_region_bob = mean_intensity_year_bob.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
max_int_region_bob = max_intensity_year_bob.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
cum_int_region_bob = cumulative_intensity_year_bob.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()

# Mean, max and cumulative intensity per year in Bay of Bengal
mean_int_region_nio = mean_intensity_year_nio.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
max_int_region_nio = max_intensity_year_nio.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()
cum_int_region_nio = cumulative_intensity_year_nio.where(ocean).weighted(w_lat).mean(dim=[latn, lonn], skipna= True).compute()


In [ ]:
# Per decade MHW Intensity bar graphs with theil-sen trends
def plot_trendlines(ax, x,y, line_color,region_name):
    
    # Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$), ${region_name}$"
    # plot the trend line 
    ax.plot(x, line_ts, color= line_color, linestyle= '--', linewidth= 2, label= label_ts)
    
    # add legends
    ax.legend()
    
# Plotting the bar graphs

years_mean_int = (mean_int_region_as.coords.get("year", None).values
              if "year" in mean_int_region_as.coords
              else mean_int_region_as.get_index("year").values).astype(int)

# combined bar graphs of mean intensities
x= years_mean_int
w= 0.25
fig, ax1 = plt.subplots(figsize=(14,6))
# add trend line 
plot_trendlines(ax1, years_mean_int, mean_int_region_as, "blue", "AS")
plot_trendlines(ax1, years_mean_int, mean_int_region_bob, "red", "BoB")
plot_trendlines(ax1, years_mean_int, mean_int_region_nio, "orange", "NIO")
ax1.bar(x - w, mean_int_region_as,  width=w, color= "#440154",  label="Arabian Sea")    #Dark Purple
ax1.bar(x,     mean_int_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # Teal
ax1.bar(x + w, mean_int_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # Yellow
ax1.set_title(f"Mean MHW Intensities in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax1, years_mean_int)
# Extract the current legend handles and labels
handles, labels = ax1.get_legend_handles_labels()
# Reorder them: [Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO]
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax1.legend(ordered_handles, ordered_labels, loc='lower left', ncol=2, framealpha=0.9, fontsize=10)
ax1.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# combined bar graphs of maximum intensities
x= years_mean_int
w= 0.25
fig, ax2 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax2, years_mean_int, max_int_region_as, "blue", "AS")
plot_trendlines(ax2, years_mean_int, max_int_region_bob, "red", "BoB")
plot_trendlines(ax2, years_mean_int, max_int_region_nio, "orange", "NIO")
ax2.bar(x - w, max_int_region_as,  width=w, color= "#440154",  label="Arabian Sea")    #Dark Purple
ax2.bar(x,     max_int_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # Teal
ax2.bar(x + w, max_int_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # Yellow
ax2.set_title(f"Maximum MHW Intensities in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold")
ax2.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax2, years_mean_int)
# Extract the current legend handles and labels
handles, labels = ax2.get_legend_handles_labels()
# Reorder them: [Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO]
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax2.legend(ordered_handles, ordered_labels, loc='lower left', ncol=2, framealpha=0.9, fontsize=10)
ax2.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# combined bar graphs of Cumulative intensities
x= years_mean_int
w= 0.25
fig, ax3 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax3, years_mean_int, cum_int_region_as, "blue", "AS")
plot_trendlines(ax3, years_mean_int, cum_int_region_bob, "red", "BoB")
plot_trendlines(ax3, years_mean_int, cum_int_region_nio, "orange", "NIO")
ax3.bar(x - w, cum_int_region_as,  width=w, color= "#440154",  label="Arabian Sea")    #Dark Purple
ax3.bar(x,     cum_int_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # Teal
ax3.bar(x + w, cum_int_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # Yellow
ax3.set_title(f"Cumulative MHW Intensities in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax3.set_xlabel("Year", fontweight= "bold")
ax3.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax3, years_mean_int)
# Extract the current legend handles and labels
handles, labels = ax3.get_legend_handles_labels()
# Reorder them: [Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO]
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax3.legend(ordered_handles, ordered_labels, loc='upper left', ncol=2, framealpha=0.9, fontsize=10)
ax3.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Per decade MHW Intensity bar graphs with poisson GLM trends
def plot_trendlines(ax, x,y, line_color,region_name):
    y= np.asarray(y)
    # Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per year.
    slope_coef = glm_results.params[1]
    year_pct_increase = (np.exp(slope_coef*10) - 1) * 100
    label_glm = f"Poisson GLM: +{year_pct_increase:.1f}% /decade ($p={p_glm_str}$), ${region_name}$"
    ax.plot(x, line_glm, color= line_color, linestyle= '--', linewidth= 2, label= label_glm)
    ax.legend()

# Plotting the bar graphs

years_mean_int = (mean_int_region_as.coords.get("year", None).values
              if "year" in mean_int_region_as.coords
              else mean_int_region_as.get_index("year").values).astype(int)

# combined bar graphs of mean intensities
x= years_mean_int
w= 0.25
fig, ax1 = plt.subplots(figsize=(14,6))
# add trend line 
plot_trendlines(ax1, years_mean_int, mean_int_region_as, "blue", "AS")
plot_trendlines(ax1, years_mean_int, mean_int_region_bob, "red", "BoB")
plot_trendlines(ax1, years_mean_int, mean_int_region_nio, "orange", "NIO")
ax1.bar(x - w, mean_int_region_as,  width=w, color= "#440154",  label="Arabian Sea")    #Dark Purple
ax1.bar(x,     mean_int_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # Teal
ax1.bar(x + w, mean_int_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # Yellow
ax1.set_title(f"Mean MHW Intensities in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
ax1.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax1, years_mean_int)
# Extract the current legend handles and labels
handles, labels = ax1.get_legend_handles_labels()
# Reorder them: [Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO]
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax1.legend(ordered_handles, ordered_labels, loc='lower left', ncol=2, framealpha=0.9, fontsize=10)
ax1.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# combined bar graphs of maximum intensities
x= years_mean_int
w= 0.25
fig, ax2 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax2, years_mean_int, max_int_region_as, "blue", "AS")
plot_trendlines(ax2, years_mean_int, max_int_region_bob, "red", "BoB")
plot_trendlines(ax2, years_mean_int, max_int_region_nio, "orange", "NIO")
ax2.bar(x - w, max_int_region_as,  width=w, color= "#440154",  label="Arabian Sea")    #Dark Purple
ax2.bar(x,     max_int_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # Teal
ax2.bar(x + w, max_int_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # Yellow
ax2.set_title(f"Maximum MHW Intensities in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold")
ax2.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax2, years_mean_int)
# Extract the current legend handles and labels
handles, labels = ax2.get_legend_handles_labels()
# Reorder them: [Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO]
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax2.legend(ordered_handles, ordered_labels, loc='lower left', ncol=2, framealpha=0.9, fontsize=10)
ax2.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# combined bar graphs of Cumulative intensities
x= years_mean_int
w= 0.25
fig, ax3 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax3, years_mean_int, cum_int_region_as, "blue", "AS")
plot_trendlines(ax3, years_mean_int, cum_int_region_bob, "red", "BoB")
plot_trendlines(ax3, years_mean_int, cum_int_region_nio, "orange", "NIO")
ax3.bar(x - w, cum_int_region_as,  width=w, color= "#440154",  label="Arabian Sea")    #Dark Purple
ax3.bar(x,     cum_int_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # Teal
ax3.bar(x + w, cum_int_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # Yellow
ax3.set_title(f"Cumulative MHW Intensities in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax3.set_xlabel("Year", fontweight= "bold")
ax3.set_ylabel("Intensity (°C)", fontweight= "bold")
apply_year_ticks(ax3, years_mean_int)
# Extract the current legend handles and labels
handles, labels = ax3.get_legend_handles_labels()
# order the labels: Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax3.legend(ordered_handles, ordered_labels, loc='upper left', ncol=2, framealpha=0.9, fontsize=10)
ax3.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# The following Script compute and plot the  combined bar graph total MHW events and days of AS, BoB and NIO regions

In [ ]:
# This Script compute and plot the combined bar of MHW events and days and graph of AS, BoB and NIO regions
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# Defire the time period
BASELINE = (1982, 2024)

# Set the regions
ROI_DICT = {
    "Arabian Sea": {"lon_min": 40.0, "lon_max": 80.0, "lat_min":  0.0, "lat_max": 30.0, "slug": "arabian_sea"},
    "Bay Of Bengal": {"lon_min": 80.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0, "slug": "bay_of_bengal"},
    "North Indian Ocean": {"lon_min": 40.0, "lon_max": 110.0,"lat_min":  0.0, "lat_max":  30.0,"slug": "north_indian_ocean"},
}
REG = "Arabian Sea"
ROI = ROI_DICT[REG]

# load the zarr file paths for event analysis
ds_event_as  = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_events_per_year_grid.zarr")
ds_event_bob = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_events_per_year_grid.zarr")
ds_event_nio = xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_events_per_year_grid.zarr")  #provide the proper path here

# load the zarr file paths for event analysis
ds_days_as  = xr.open_zarr("/home/Desktop/Jupyter files/arabian_sea/mhw_days_per_year_grid.zarr")
ds_days_bob = xr.open_zarr("/home/Desktop/Jupyter files/bay_of_bengal/mhw_days_per_year_grid.zarr")
ds_days_nio = xr.open_zarr("/home/Desktop/Jupyter files/north_indian_ocean/mhw_days_per_year_grid.zarr")  #provide the proper path here

# Corrected coordinate extraction
latn = "lat" if "lat" in ds_mean_int.coords else "latitude"
lonn = "lon" if "lon" in ds_mean_int.coords else "longitude"

# Extract the event variable array from the loaded dataset of Arabian Sea 
events_per_year_as = ds_event_as["events_per_year"]
events_per_year_bob = ds_event_bob["events_per_year"]
events_per_year_nio = ds_event_nio["events_per_year"]

# Extract the days variable array from the loaded dataset of Arabian Sea 
days_per_year_as  =  ds_days_as["days_per_year"]
days_per_year_bob  =  ds_days_bob["days_per_year"]
days_per_year_nio  =  ds_days_nio["days_per_year"]


# X-axis year tick
YEAR_TICK_STEP  = 3
YEAR_TICK_START = None
YEAR_TICK_END   = None
YEAR_TICK_ROT   = 0

# Area Weights Calculation
def area_weights_1d(ds_event_path: xr.Dataset, latn: str) -> xr.DataArray:
    return np.cos(np.deg2rad(ds_event_path[latn]))

def apply_year_ticks(ax, years, step=YEAR_TICK_STEP, start=YEAR_TICK_START, end=YEAR_TICK_END, rotate=YEAR_TICK_ROT):
    years = np.asarray(years, dtype=int)
    y_min, y_max = int(years.min()), int(years.max())
    a = y_min if start is None else int(start)
    b = y_max if end   is None else int(end)
    if (step is None) or (step == 0):
        ticks = np.unique(years)
    else:
        ticks = np.arange(a, b + 1, int(step), dtype=int)
    ax.set_xticks(ticks)
    ax.set_xticklabels([str(y) for y in ticks], rotation=rotate)

# calculte the Basin area avarage
# Calculate weights
w_lat = area_weights_1d(ds_event_path, latn)

# Area-weighted regional means of events (per year)
freq_region_as = (events_per_year_as.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
freq_region_bob = (events_per_year_bob.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
freq_region_nio = (events_per_year_nio.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

# Area-weighted regional means of days(per year)
days_region_as = (days_per_year_as.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
days_region_bob = (days_per_year_bob.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()
days_region_nio = (days_per_year_nio.where(ocean)).weighted(w_lat).mean(dim=[latn, lonn]).compute()

# Plot the graphs
years_freq_as = (freq_region_as.coords.get("year", None).values
              if "year" in freq_region_as.coords
              else freq_region_as.get_index("year").values).astype(int)

years_days_as = (days_region_as.coords.get("year", None).values
              if "year" in days_region_as.coords
              else days_region_as.get_index("year").values).astype(int)

In [ ]:
# Per decade MHW Intensity bar graphs with Theil-sen trends
def plot_trendlines(ax, x,y, line_color,region_name):
    
    # Theil-Sen non-parametric trend
    ts_res= stats.mstats.theilslopes(y,x, 0.95)
    slop_ts= ts_res[0]
    intercept_ts= ts_res[1]
    line_ts = (slop_ts*x) + intercept_ts
    # mann-kendall significance test 
    tau, p_ts = stats.kendalltau(x,y)
    # p value test
    p_ts_str = "<0.01" if p_ts < 0.01 else f"{p_ts:.3f}"
    label_ts =  f"Theil-Sen Trend: +{slop_ts*10:.2f}/decade ($p={p_ts_str}$), ${region_name}$"
    # plot the trend line 
    ax.plot(x, line_ts, color= line_color, linestyle= '--', linewidth= 2, label= label_ts)
    
    # add legends
    ax.legend()

# Combine the bar graphs of ,mhw events in the regions
x = years_freq_as
w = 0.25   # adjust the bar width

# Plot the figure
fig, ax1 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax1, years_freq_as, freq_region_as, "blue", "AS")
plot_trendlines(ax1, years_freq_as, freq_region_bob, "red", "BoB")
plot_trendlines(ax1, years_freq_as, freq_region_nio, "orange", "NIO")
ax1.bar(x - w, freq_region_as,  width=w, color="#0072B2",  label="Arabian Sea")   # blue 
ax1.bar(x,     freq_region_bob, width=w, color= "#D55E00", label="Bay of Bengal")  # vermillion
ax1.bar(x + w, freq_region_nio, width=w, color= "#009E73", label="North Indian Ocean")  # bluish green
ax1.set_title(f"MHW Events Per Year in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_ylabel("Events per year", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
apply_year_ticks(ax1, years_freq_as)
# Extract the current legend handles and labels
handles, labels = ax1.get_legend_handles_labels()
# Reorder them: [Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO]
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax1.legend(ordered_handles, ordered_labels, loc='upper left', ncol=2, framealpha=0.9, fontsize=10)
ax1.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Plot the figure
fig, ax2 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax2, years_days_as, days_region_as, "blue", "AS")
plot_trendlines(ax2, years_days_as, days_region_bob, "red", "BoB")
plot_trendlines(ax2, years_days_as, days_region_nio, "orange", "NIO")
ax2.bar(x - w, days_region_as,  width=w, color= "#440154",  label="Arabian Sea")   # blue 
ax2.bar(x,     days_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # vermillion
ax2.bar(x + w, days_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # bluish green
ax2.set_title(f"MHW days Per Year in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_ylabel("Days per year", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold")
apply_year_ticks(ax2, years_days_as)
# Extract the current legend handles and labels
handles, labels = ax2.get_legend_handles_labels()
# order the labels: Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax2.legend(ordered_handles, ordered_labels, loc='upper left', ncol=2, framealpha=0.9, fontsize=10)
ax2.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()



In [ ]:
# Per decade MHW Intensity bar graphs with poisson GLM trends
def plot_trendlines(ax, x,y, line_color,region_name):
    y= np.asarray(y)
    # Poisson generalized linear model (GLM) trend 
    # Statsmodels requires explicitly add a constant for the intercept
    X_glm = sm.add_constant(x)
    glm_model = sm.GLM(y, X_glm, family=sm.families.Poisson())
    glm_results = glm_model.fit()
    # p value test
    p_glm = glm_results.pvalues[1]
    p_glm_str = "< 0.01" if p_glm < 0.01 else f"{p_glm:.3f}"
    # plot the trend line
    line_glm = glm_results.predict(X_glm)
    
    # The GLM slop will be in percentage change per year.
    slope_coef = glm_results.params[1]
    year_pct_increase = (np.exp(slope_coef*10) - 1) * 100
    label_glm = f"Poisson GLM: +{year_pct_increase:.1f}% /decade ($p={p_glm_str}$), ${region_name}$"
    ax.plot(x, line_glm, color= line_color, linestyle= '--', linewidth= 2, label= label_glm)
    ax.legend()


# Combine the bar graphs of ,mhw events in the regions
x = years_freq_as
w = 0.25   # adjust the bar width

# Plot the figure
fig, ax1 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax1, years_freq_as, freq_region_as, "blue", "AS")
plot_trendlines(ax1, years_freq_as, freq_region_bob, "red", "BoB")
plot_trendlines(ax1, years_freq_as, freq_region_nio, "orange", "NIO")
ax1.bar(x - w, freq_region_as,  width=w, color="#0072B2",  label="Arabian Sea")   # blue 
ax1.bar(x,     freq_region_bob, width=w, color= "#D55E00", label="Bay of Bengal")  # vermillion
ax1.bar(x + w, freq_region_nio, width=w, color= "#009E73", label="North Indian Ocean")  # bluish green
ax1.set_title(f"MHW Events Per Year in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax1.set_ylabel("Events per year", fontweight= "bold")
ax1.set_xlabel("Year", fontweight= "bold")
apply_year_ticks(ax1, years_freq_as)
# Extract the current legend handles and labels
handles, labels = ax1.get_legend_handles_labels()
# Reorder them: [Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO]
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax1.legend(ordered_handles, ordered_labels, loc='upper left', ncol=2, framealpha=0.9, fontsize=10)
ax1.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()

# Plot the figure
fig, ax2 = plt.subplots(figsize=(14,6))
# add trend line
plot_trendlines(ax2, years_days_as, days_region_as, "blue", "AS")
plot_trendlines(ax2, years_days_as, days_region_bob, "red", "BoB")
plot_trendlines(ax2, years_days_as, days_region_nio, "orange", "NIO")
ax2.bar(x - w, days_region_as,  width=w, color= "#440154",  label="Arabian Sea")   # blue 
ax2.bar(x,     days_region_bob, width=w, color= "#21918c", label="Bay of Bengal")  # vermillion
ax2.bar(x + w, days_region_nio, width=w, color= "#fde725", label="North Indian Ocean")  # bluish green
ax2.set_title(f"MHW days Per Year in AS, BoB and NIO Regions ({BASELINE[0]}–{BASELINE[1]})", fontweight= "bold")
ax2.set_ylabel("Days per year", fontweight= "bold")
ax2.set_xlabel("Year", fontweight= "bold")
apply_year_ticks(ax2, years_days_as)
# Extract the current legend handles and labels
handles, labels = ax2.get_legend_handles_labels()
# order the labels: Line AS, Bar AS, Line BoB, Bar BoB, Line NIO, Bar NIO
ordered_handles = [handles[0], handles[1], handles[2], handles[3], handles[4], handles[5]]
ordered_labels  = [labels[0], labels[1], labels[2], labels[3], labels[4], labels[5]]
ax2.legend(ordered_handles, ordered_labels, loc='upper left', ncol=2, framealpha=0.9, fontsize=10)
ax2.grid(True, ls="--", alpha=0.4)
plt.tight_layout()
plt.show()